# EXP-006 판정 근거: KURE-Reranker-nano(GPU) 재정렬

EXP-004 대비 dev-full 지표, 정답 순위 이동, 층별 차이, 지연과 운영 비용을 보고 채택 여부와 배포 지연 상한(SPEC 미결 3)을 정한다. 질의 원문은 출력하지 않는다.

In [1]:
import json
from pathlib import Path

import pandas as pd

from rag.eval.metrics import gold_ranks, load_cfg, load_gold, read_run

In [2]:
ROOT = Path.cwd().parent
RUNS = ROOT / 'data' / 'runs'
RERANK = ROOT / 'data' / 'rerank'
BASE_RUN = RUNS / 'kure-v1-fixed-512-64-hybrid-cond-filter-dev-full.csv'
CAND_RUN = RUNS / 'kure-v1-fixed-512-64-hybrid-cond-filter-ce-dev-full.csv'
JEV_RUN = RUNS / 'kure-v1-fixed-512-64-hybrid-cond-filter-rerank-dev-full.csv'
PAIRS = RERANK / 'summary_q-full.pairs.jsonl'
# 운영 비용 가정: HF Inference Endpoints 요금(공식 pricing 문서, 2026-10-13 확인), Jev는 EXP-005 실측
T4_USD_PER_HOUR = 0.50
HOURS_PER_MONTH = 730
JEV_USD_PER_QUERY = 2.015 / 3016

## 전체 지표 (판정)

In [3]:
single = json.loads((RUNS / 'exp006-single-compare.json').read_text(encoding='utf-8'))
print(pd.DataFrame({'EXP-004': single['base']['metrics'], 'EXP-006': single['cand']['metrics']}).round(4))
print('Recall@5 차이', round(single['diff'] * 100, 2), '%p, 95% 구간', [round(c * 100, 2) for c in single['ci']], single['verdict'])

           EXP-004  EXP-006
recall@1    0.7954   0.8916
recall@5    0.9164   0.9473
recall@10   0.9450   0.9589
mrr@10      0.8488   0.9161
Recall@5 차이 3.08 %p, 95% 구간 [2.29, 3.86] 채택


## 정답 순위 이동

기준 순위는 RRF 상위 20개 후보 쌍 파일(필터 전)에서, 후보 순위는 EXP-006 순위 파일(상위 10개)에서 잰다.

In [4]:
gold = load_gold(load_cfg(), 'dev-full')
pairs = pd.read_json(PAIRS, lines=True)[['qid', 'doc_id']]
pairs['rank'] = pairs.groupby('qid').cumcount() + 1
rrf20 = pairs.merge(gold['doc_id'].reset_index(), on=['qid', 'doc_id']).set_index('qid')['rank'].reindex(gold.index)
base = gold_ranks(read_run(BASE_RUN), gold, 10)
cand = gold_ranks(read_run(CAND_RUN), gold, 10)
jev = gold_ranks(read_run(JEV_RUN), gold, 10)
print(len(gold), 'queries / RRF 상위 20 안 정답', int(rrf20.notna().sum()))

3016 queries / RRF 상위 20 안 정답 2914


In [5]:
def bucket(r, edges=((1, '1'), (5, '2~5'), (10, '6~10'), (20, '11~20'))):
    return r.map(lambda x: next((n for e, n in edges if x <= e), '밖') if pd.notna(x) else '밖')

print('행: RRF(필터 전) 순위, 열: EXP-006 순위')
print(pd.crosstab(bucket(rrf20), bucket(cand)).reindex(index=['1', '2~5', '6~10', '11~20', '밖'], columns=['1', '2~5', '6~10', '밖'], fill_value=0))

행: RRF(필터 전) 순위, 열: EXP-006 순위
rank      1  2~5  6~10    밖
rank                       
1      2312   75    13    2
2~5     278   70     9    6
6~10     60   11     9    5
11~20    39   12     4    9
밖         0    0     0  102


In [6]:
rows = []
for name, r in [('EXP-006 KURE-nano', cand), ('EXP-005 Jev', jev)]:
    for k in (1, 5, 10):
        rows.append({'reranker': name, 'k': k,
                     '진입': int(((base.isna() | (base > k)) & (r <= k)).sum()),
                     '이탈': int(((base <= k) & (r.isna() | (r > k))).sum())})
print(pd.DataFrame(rows).pivot(index='k', columns='reranker', values=['진입', '이탈']))

                  진입                            이탈                  
reranker EXP-005 Jev EXP-006 KURE-nano EXP-005 Jev EXP-006 KURE-nano
k                                                                   
1                226               377         581                87
5                 93               122         186                29
10                49                55          65                13


## 층별 Recall@5

In [8]:
df = gold.assign(base5=base <= 5, cand5=cand <= 5)
for col in ('meeting_name', 'qna_type'):
    t = df.groupby(col).agg(n=('base5', 'size'), base=('base5', 'mean'), cand=('cand5', 'mean'))
    t['diff_%p'] = (t['cand'] - t['base']) * 100
    print(t.round(4).sort_values('n', ascending=False))
    print()

                 n    base    cand  diff_%p
meeting_name                               
국정감사          1225  0.9249  0.9584   3.3469
소위원회          1214  0.9267  0.9481   2.1417
특별위원회          307  0.8730  0.9121   3.9088
예산결산특별위원회      158  0.8797  0.9430   6.3291
본회의            112  0.8839  0.9196   3.5714

             n    base    cand  diff_%p
qna_type                               
추출형       2727  0.9208  0.9490   2.8236
단답형        289  0.8754  0.9308   5.5363



## multi-doc (관찰)

In [9]:
multi = json.loads((RUNS / 'exp006-multi-compare.json').read_text(encoding='utf-8'))
print(pd.DataFrame({'EXP-004': multi['base']['metrics'], 'EXP-006': multi['cand']['metrics']}).round(4))

             EXP-004  EXP-006
recall@5      0.9361   0.9348
complete@5    0.8717   0.8717
hit@5         0.9883   0.9883
recall@10     0.9597   0.9579
complete@10   0.9217   0.9183
hit@10        0.9917   0.9917


## 지연 (질의당 20쌍, 초)

CPU는 EXP-005 설계 단계 측정(2스레드, dev-small 질의 20개), Jev는 EXP-005 dev-full(OpenRouter 왕복 포함, 동시 요청 8개), T4는 이번 측정(GPU 계산만, 질의 100개, 하나씩)이다. T4 값에는 추론 서버의 네트워크 왕복과 기동 지연이 없다.

In [10]:
cpu = json.loads((RERANK / 'latency-summary_q-dev-small-cpu.json').read_text(encoding='utf-8'))
t4 = json.loads((RERANK / 'latency-summary_q-full-cross-encoder.json').read_text(encoding='utf-8'))
jv = json.loads((RERANK / 'latency-summary_q-full-jev.json').read_text(encoding='utf-8'))
stats = ('mean', 'p50', 'p95', 'max')
lat = pd.DataFrame({
    'CPU 2스레드 KURE-nano': {k: cpu[k] for k in stats},
    'Jev OpenRouter': {k: jv[f'latency_{k}'] for k in stats},
    'T4 fp16 KURE-nano': {k: t4[k] for k in stats},
})
print(lat.round(3))

      CPU 2스레드 KURE-nano  Jev OpenRouter  T4 fp16 KURE-nano
mean              37.119           0.599              0.379
p50               36.032           0.532              0.377
p95               41.234           0.835              0.410
max               41.732           5.882              0.426


## 운영 비용

T4 전용 엔드포인트는 켜진 시간으로, Jev는 질의 수로 과금된다. T4 한 대의 처리 상한은 질의를 하나씩 처리할 때 1 / 평균 지연이다.

In [11]:
t4_month = T4_USD_PER_HOUR * HOURS_PER_MONTH
qps = 1 / t4['mean']
breakeven = t4_month / JEV_USD_PER_QUERY
print(f'T4 상시 가동 월 ${t4_month:.0f}, 처리 상한 약 {qps:.1f} 질의/초(월 {qps * 3600 * HOURS_PER_MONTH / 1e6:.1f}M 질의)')
print(f'Jev 질의당 ${JEV_USD_PER_QUERY:.5f}, T4 상시 가동과 비용이 같아지는 월 질의 수 약 {breakeven / 1e3:.0f}K')
for q in (1_000, 10_000, 100_000, 1_000_000):
    print(f'월 {q:>9,} 질의: Jev ${q * JEV_USD_PER_QUERY:8.2f} / T4 상시 ${t4_month:.0f}')

T4 상시 가동 월 $365, 처리 상한 약 2.6 질의/초(월 6.9M 질의)
Jev 질의당 $0.00067, T4 상시 가동과 비용이 같아지는 월 질의 수 약 546K
월     1,000 질의: Jev $    0.67 / T4 상시 $365
월    10,000 질의: Jev $    6.68 / T4 상시 $365
월   100,000 질의: Jev $   66.81 / T4 상시 $365
월 1,000,000 질의: Jev $  668.10 / T4 상시 $365
